# Similarity

Analyze cosine similarity between the final prompt-token representations of UniEdit edit, generality, and locality prompts. The notebook filters records by complete-target top-1 correctness, saves record-level results to `data/similarity/{model}.json`, and plots aggregate mean with one-standard-deviation bands.

## Environment And Configuration

In [ ]:
from __future__ import annotations

import gc
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from datasets import load_dataset
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE_CONFIG_DIR = PROJECT_ROOT / "src" / "config" / "base"
DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "similarity"
DATASET_SPLIT = "data"
OUTPUT_DIR = PROJECT_ROOT / "data" / "similarity"


def similarity_output_path(record_limit: int | None) -> Path:
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    output_stem = MODEL_KEY if record_limit is None else f"{MODEL_KEY}.limit-{int(record_limit)}"
    return OUTPUT_DIR / f"{output_stem}.json"


def similarity_failed_output_path(record_limit: int | None) -> Path:
    output_path = similarity_output_path(record_limit)
    return output_path.with_name(f"{output_path.stem}-failed.json")


print(f"output_dir={OUTPUT_DIR}")

## Load Dataset And Base Model

In [ ]:
base_config = None
model_source = None
loaded_model_key = None
dataset = None
model = None
tokenizer = None
backbone = None
output_head = None
decoder_layers = None
input_device = None
output_device = None


def load_similarity_resources() -> None:
    global base_config, model_source, loaded_model_key
    global dataset, model, tokenizer, backbone, output_head
    global decoder_layers, input_device, output_device

    if model is not None:
        if loaded_model_key == MODEL_KEY:
            return
        release_similarity_resources()

    base_config_path = BASE_CONFIG_DIR / f"{MODEL_KEY}.yaml"
    if not base_config_path.exists():
        raise FileNotFoundError(f"Base model config does not exist: {base_config_path}")
    with open(base_config_path, encoding="utf-8") as file:
        base_config = yaml.safe_load(file)
    model_source = base_config["model_name"]
    loaded_model_key = MODEL_KEY
    print(f"model={MODEL_KEY} checkpoint={model_source}")

    dataset = load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    model_parallel = bool(base_config.get("model_parallel", False))
    model_dtype = torch.bfloat16 if base_config.get("fp16", True) else torch.float32
    model_kwargs = {"dtype": model_dtype}
    if model_parallel:
        model_kwargs["device_map"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(model_source, **model_kwargs)
    if not model_parallel:
        device = torch.device(
            f"cuda:{int(base_config.get('gpu_id', 0))}"
            if torch.cuda.is_available()
            else "cpu"
        )
        model = model.to(device)
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(model_source)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model_prefix = getattr(model, "base_model_prefix", None)
    backbone = getattr(model, base_model_prefix, None) if base_model_prefix else None
    if backbone is None or backbone is model:
        backbone = model.base_model
    if backbone is model:
        raise RuntimeError("Could not resolve the causal language model backbone.")

    output_head = model.get_output_embeddings()
    decoder_layers = getattr(backbone, "layers", None)
    if decoder_layers is None:
        raise RuntimeError(f"Unsupported backbone without .layers: {type(backbone).__name__}")

    input_device = model.get_input_embeddings().weight.device
    output_device = next(output_head.parameters()).device
    print(f"dataset_rows={len(dataset)} layers={len(decoder_layers)} input_device={input_device}")


def release_similarity_resources() -> None:
    global base_config, model_source, loaded_model_key
    global dataset, model, tokenizer, backbone, output_head
    global decoder_layers, input_device, output_device

    base_config = None
    model_source = None
    loaded_model_key = None
    dataset = None
    model = None
    tokenizer = None
    backbone = None
    output_head = None
    decoder_layers = None
    input_device = None
    output_device = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Prompt Filtering

A record is retained only when every token of at least one accepted target or alias is the model's top-1 prediction under teacher forcing for its edit, generality, and locality prompts. Optional failed-record logging also saves a short greedy continuation for each prompt so failures can be inspected directly. Eligibility is model-specific, so cross-model comparisons should use the saved `source_index` values to restrict results to shared eligible records.

In [ ]:
ROLE_ORDER = ("edit", "generality", "locality")
ROLE_TO_PROMPT_ID = {role: f"p{index}" for index, role in enumerate(ROLE_ORDER, start=1)}
PAIR_SPECS = (
    ("p1-p2", 0, 1, "edit - generality"),
    ("p1-p3", 0, 2, "edit - locality"),
    ("p2-p3", 1, 2, "generality - locality"),
)
SELECTION_CRITERION = (
    "every token of at least one accepted target is top-1 under teacher forcing"
)


def target_continuation(prompt: str, target: str) -> str:
    if not target:
        raise ValueError(f"Empty expected target for prompt: {prompt!r}")
    if prompt and prompt[-1].isspace():
        return target.lstrip()
    return target if target[0].isspace() else f" {target}"


def target_token_ids(prompt: str, target: str) -> list[int]:
    token_ids = tokenizer.encode(
        target_continuation(prompt, target),
        add_special_tokens=False,
    )
    if not token_ids:
        raise ValueError(f"Expected target produced no tokens: {target!r}")
    return [int(token_id) for token_id in token_ids]


def build_filter_candidates(
    dataset,
    record_limit: int | None,
) -> tuple[list[dict], dict[int, dict], list[dict]]:
    source_count = len(dataset) if record_limit is None else min(int(record_limit), len(dataset))
    candidates: list[dict] = []
    contexts: dict[int, dict] = {}
    target_checks: list[dict] = []

    for source_index in range(source_count):
        record = dataset[source_index]
        contexts[source_index] = {"source_index": source_index}

        for role in ROLE_ORDER:
            prompt = record[role]["prompt"]
            target = record[role]["target"]
            aliases = list(record[role].get("aliases", []))
            prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
            if not prompt_ids:
                raise ValueError(f"Prompt produced no tokens: {prompt!r}")
            candidate_index = len(candidates)
            candidates.append({
                "source_index": source_index,
                "role": role,
                "prompt": prompt,
                "target": target,
                "aliases": aliases,
                "checked_target_count": 1 + len(aliases),
                "matched_target": None,
            })
            for accepted_target in (target, *aliases):
                target_checks.append({
                    "candidate_index": candidate_index,
                    "target": accepted_target,
                    "prompt_ids": prompt_ids,
                    "target_ids": target_token_ids(prompt, accepted_target),
                })

    return candidates, contexts, target_checks


def verify_target_checks(target_checks: list[dict]) -> list[bool]:
    sequences = [
        check["prompt_ids"] + check["target_ids"]
        for check in target_checks
    ]
    batch = tokenizer.pad(
        {"input_ids": sequences},
        padding=True,
        return_attention_mask=True,
        return_tensors="pt",
    ).to(input_device)

    with torch.inference_mode():
        outputs = backbone(**batch, use_cache=False, return_dict=True)
        hidden_states = outputs.last_hidden_state

        row_indices = []
        token_positions = []
        expected_ids = []
        check_spans = []
        padded_length = batch["input_ids"].shape[1]
        for row_index, (sequence, check) in enumerate(zip(sequences, target_checks)):
            target_ids = check["target_ids"]
            left_padding = padded_length - len(sequence)
            first_prediction_position = left_padding + len(check["prompt_ids"]) - 1
            span_start = len(expected_ids)
            for offset, expected_id in enumerate(target_ids):
                row_indices.append(row_index)
                token_positions.append(first_prediction_position + offset)
                expected_ids.append(expected_id)
            check_spans.append((span_start, len(expected_ids)))

        selected_hidden = hidden_states[
            torch.tensor(row_indices, device=hidden_states.device),
            torch.tensor(token_positions, device=hidden_states.device),
        ].to(output_device)
        predicted_ids = output_head(selected_hidden).float().argmax(dim=-1).cpu().tolist()

    results = [
        predicted_ids[start:end] == expected_ids[start:end]
        for start, end in check_spans
    ]
    del batch, outputs, hidden_states, selected_hidden, predicted_ids
    return results


def normalize_match_text(value: str) -> str:
    return " ".join(
        "".join(character.casefold() if character.isalnum() else " " for character in value).split()
    )


def find_generated_match(generated_answer: str, candidate: dict) -> str | None:
    normalized_answer = normalize_match_text(generated_answer)
    padded_answer = f" {normalized_answer} "
    for accepted_target in (candidate["target"], *candidate["aliases"]):
        normalized_target = normalize_match_text(accepted_target)
        if normalized_target and f" {normalized_target} " in padded_answer:
            return accepted_target
    return None


def attach_generated_answers(
    candidates: list[dict],
    batch_size: int,
    max_new_tokens: int,
) -> None:
    if max_new_tokens <= 0:
        raise ValueError("failed_generation_max_new_tokens must be positive.")
    for start in tqdm(
        range(0, len(candidates), batch_size),
        desc="Generate failed answers",
    ):
        candidate_batch = candidates[start : start + batch_size]
        batch = tokenizer(
            [candidate["prompt"] for candidate in candidate_batch],
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(input_device)
        prompt_width = batch["input_ids"].shape[1]

        with torch.inference_mode():
            generated_ids = model.generate(
                **batch,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.pad_token_id,
            )
        generated_answers = tokenizer.batch_decode(
            generated_ids[:, prompt_width:],
            skip_special_tokens=True,
        )
        for candidate, generated_answer in zip(candidate_batch, generated_answers):
            generated_answer = generated_answer.strip()
            generated_match = find_generated_match(generated_answer, candidate)
            candidate["generated_answer"] = generated_answer
            candidate["generated_match"] = generated_match
            candidate["generation_matches_expected"] = generated_match is not None

        del batch, generated_ids


def select_eligible_records(
    dataset,
    record_limit: int | None,
    filter_batch_size: int,
    collect_failed_records: bool,
    failed_generation_max_new_tokens: int,
) -> tuple[list[dict], list[dict], dict]:
    if filter_batch_size <= 0:
        raise ValueError("filter_batch_size must be positive.")
    candidates, contexts, target_checks = build_filter_candidates(
        dataset, record_limit=record_limit
    )
    for start in tqdm(
        range(0, len(target_checks), filter_batch_size),
        desc="Verify complete targets",
    ):
        batch_checks = target_checks[start : start + filter_batch_size]
        for check, is_valid in zip(batch_checks, verify_target_checks(batch_checks)):
            candidate = candidates[check["candidate_index"]]
            if is_valid and candidate["matched_target"] is None:
                candidate["matched_target"] = check["target"]

    for candidate in candidates:
        candidate["is_valid"] = candidate["matched_target"] is not None

    candidates_by_record: dict[int, dict[str, list[dict]]] = defaultdict(lambda: defaultdict(list))
    total_candidate_counts = Counter()
    valid_candidate_counts = Counter()
    for candidate in candidates:
        candidates_by_record[candidate["source_index"]][candidate["role"]].append(candidate)
        total_candidate_counts[candidate["role"]] += 1
        if candidate["is_valid"]:
            valid_candidate_counts[candidate["role"]] += 1

    failed_source_indices = {
        source_index
        for source_index in contexts
        if any(
            not candidates_by_record[source_index][role][0]["is_valid"]
            for role in ROLE_ORDER
        )
    }
    if collect_failed_records:
        attach_generated_answers(
            [
                candidate
                for candidate in candidates
                if candidate["source_index"] in failed_source_indices
            ],
            batch_size=filter_batch_size,
            max_new_tokens=failed_generation_max_new_tokens,
        )

    selected_records: list[dict] = []
    failed_records: list[dict] = []
    skip_reason_counts = Counter()
    for source_index, context in contexts.items():
        valid_by_role = {
            role: [candidate for candidate in candidates_by_record[source_index][role] if candidate["is_valid"]]
            for role in ROLE_ORDER
        }
        missing_roles = [role for role in ROLE_ORDER if not valid_by_role[role]]
        if missing_roles:
            skip_reason_counts.update(missing_roles)
            if collect_failed_records:
                failed_prompts = []
                for role in ROLE_ORDER:
                    failed_prompt = dict(candidates_by_record[source_index][role][0])
                    failed_prompt["id"] = ROLE_TO_PROMPT_ID[role]
                    failed_prompt["passed"] = failed_prompt.pop("is_valid")
                    failed_prompt["failure_reason"] = (
                        None
                        if failed_prompt["passed"]
                        else "no target or alias matched the complete teacher-forced top-1 sequence"
                    )
                    failed_prompts.append(failed_prompt)
                failed_records.append({
                    **context,
                    "failed_roles": missing_roles,
                    "prompts": failed_prompts,
                })
            continue

        selected_prompts = []
        for role in ROLE_ORDER:
            selected = dict(valid_by_role[role][0])
            selected.pop("is_valid", None)
            selected["id"] = ROLE_TO_PROMPT_ID[role]
            selected_prompts.append(selected)

        selected_records.append({**context, "prompts": selected_prompts})

    report = {
        "source_records": len(contexts),
        "eligible_records": len(selected_records),
        "skipped_records": len(contexts) - len(selected_records),
        "skip_reason_counts": dict(skip_reason_counts),
        "total_candidate_counts": dict(total_candidate_counts),
        "valid_candidate_counts": dict(valid_candidate_counts),
    }
    return selected_records, failed_records, report

In [ ]:
def display_selection_report(selection_report: dict) -> None:
    display(pd.DataFrame([selection_report]))
    display(pd.DataFrame({
        "total": selection_report["total_candidate_counts"],
        "valid": selection_report["valid_candidate_counts"],
    }).rename_axis("role"))

## Layer Similarity Analysis

The embedding output is excluded. Entries represent final prompt-token residual states between decoder blocks; the final returned entry includes the model's final normalization.

In [ ]:
def analyze_record_batch(record_batch: list[dict]) -> list[dict]:
    prompts = [
        prompt_info["prompt"]
        for record in record_batch
        for prompt_info in record["prompts"]
    ]
    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    ).to(input_device)

    with torch.inference_mode():
        outputs = backbone(
            **batch,
            output_hidden_states=True,
            use_cache=False,
            return_dict=True,
        )

    n_layers = len(decoder_layers)
    activations = torch.stack([
        hidden_state[:, -1, :].float().cpu()
        for hidden_state in outputs.hidden_states[1 : n_layers + 1]
    ])
    activations = activations.reshape(n_layers, len(record_batch), len(ROLE_ORDER), -1)
    normalized = F.normalize(activations, p=2, dim=-1)
    similarities = normalized @ normalized.transpose(-1, -2)

    analyzed_records = []
    for record_offset, record in enumerate(record_batch):
        pair_similarities = {
            pair_key: [
                round(float(value), 8)
                for value in similarities[:, record_offset, left_index, right_index].tolist()
            ]
            for pair_key, left_index, right_index, _ in PAIR_SPECS
        }
        analyzed_records.append({**record, "pair_similarities": pair_similarities})

    del batch, outputs, activations, normalized, similarities
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return analyzed_records


def aggregate_records(records: list[dict]) -> dict:
    if not records:
        raise ValueError("No eligible records were analyzed.")
    pairs = {}
    for pair_key, _, _, label in PAIR_SPECS:
        values = np.asarray(
            [record["pair_similarities"][pair_key] for record in records],
            dtype=np.float64,
        )
        pairs[pair_key] = {
            "label": label,
            "mean": np.round(values.mean(axis=0), 8).tolist(),
            "std": np.round(values.std(axis=0, ddof=0), 8).tolist(),
        }
    return {"record_count": len(records), "pairs": pairs}


def write_json_atomic(payload: dict, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(f"{destination.suffix}.tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=True, allow_nan=False)
    temporary.replace(destination)


def similarity_dataset_metadata() -> dict:
    return {
        "repo": DATASET_REPO,
        "config": DATASET_CONFIG,
        "split": DATASET_SPLIT,
        "rows": len(dataset),
        "fingerprint": getattr(dataset, "_fingerprint", None),
    }


def run_similarity_analysis(
    selected_records: list[dict],
    selection_report: dict,
    output_path: Path,
    similarity_record_batch_size: int,
    overwrite_output: bool,
) -> int:
    if similarity_record_batch_size <= 0:
        raise ValueError("similarity_record_batch_size must be positive.")
    if output_path.exists() and not overwrite_output:
        raise FileExistsError(
            f"Output already exists: {output_path}. Enable overwrite to replace it."
        )

    completed_records: list[dict] = []
    for start in tqdm(
        range(0, len(selected_records), similarity_record_batch_size),
        desc="Analyze representations",
    ):
        batch_records = selected_records[start : start + similarity_record_batch_size]
        completed_records.extend(analyze_record_batch(batch_records))

    completed_records.sort(key=lambda record: record["source_index"])
    result = {
        "dataset": similarity_dataset_metadata(),
        "selection": {
            "criterion": SELECTION_CRITERION,
            **selection_report,
        },
        "prompt_roles": [
            {"id": ROLE_TO_PROMPT_ID[role], "role": role}
            for role in ROLE_ORDER
        ],
        "pair_order": [
            {"pair": pair_key, "label": label}
            for pair_key, _, _, label in PAIR_SPECS
        ],
        "records": completed_records,
        "aggregate": aggregate_records(completed_records),
    }
    write_json_atomic(result, output_path)
    return result["aggregate"]["record_count"]

In [ ]:
def calculate_similarity(
    record_limit: int | None,
    filter_batch_size: int,
    failed_generation_max_new_tokens: int,
    similarity_record_batch_size: int,
    log_failed_records: bool,
    overwrite_output: bool,
) -> None:
    output_path = similarity_output_path(record_limit)
    failed_output_path = similarity_failed_output_path(record_limit)
    if output_path.exists() and not overwrite_output:
        raise FileExistsError(
            f"Output already exists: {output_path}. Enable overwrite to replace it."
        )
    if log_failed_records and failed_output_path.exists() and not overwrite_output:
        raise FileExistsError(
            f"Failed-record output already exists: {failed_output_path}. "
            "Enable overwrite to replace it."
        )

    try:
        load_similarity_resources()
        selected_records, failed_records, selection_report = select_eligible_records(
            dataset,
            record_limit=record_limit,
            filter_batch_size=filter_batch_size,
            collect_failed_records=log_failed_records,
            failed_generation_max_new_tokens=failed_generation_max_new_tokens,
        )
        display_selection_report(selection_report)
        if log_failed_records:
            write_json_atomic(
                {
                    "dataset": similarity_dataset_metadata(),
                    "selection": {
                        "criterion": SELECTION_CRITERION,
                        **selection_report,
                    },
                    "records": failed_records,
                },
                failed_output_path,
            )
            print(f"saved_failed={failed_output_path} records={len(failed_records)}")
        record_count = run_similarity_analysis(
            selected_records,
            selection_report=selection_report,
            output_path=output_path,
            similarity_record_batch_size=similarity_record_batch_size,
            overwrite_output=overwrite_output,
        )
        print(f"saved={output_path} records={record_count}")
    finally:
        release_similarity_resources()

## Aggregate Table And Plot

In [ ]:
PAIR_COLORS = {
    "p1-p2": "#0072B2",
    "p1-p3": "#E69F00",
    "p2-p3": "#009E73",
}


def result_layer_count(result: dict) -> int:
    pair_lengths = {
        len(values[stat_name])
        for values in result["aggregate"]["pairs"].values()
        for stat_name in ("mean", "std")
    }
    if len(pair_lengths) != 1:
        raise ValueError(f"Inconsistent aggregate layer counts: {sorted(pair_lengths)}")
    n_layers = pair_lengths.pop()
    if n_layers <= 0:
        raise ValueError("Similarity result contains no transformer layers.")
    return n_layers


def resolve_plot_layers(result: dict, layer_range: tuple[int, int] | None) -> np.ndarray:
    n_layers = result_layer_count(result)
    if layer_range is None:
        return np.arange(n_layers)
    if len(layer_range) != 2:
        raise ValueError("layer_range must be None or a (start, end) tuple.")
    start_layer, end_layer = (int(value) for value in layer_range)
    if not 0 <= start_layer <= end_layer < n_layers:
        raise ValueError(
            f"Invalid inclusive layer range {layer_range}; model has layers 0..{n_layers - 1}."
        )
    return np.arange(start_layer, end_layer + 1)


def resolve_plot_pairs(pairs: list[str] | None) -> list[tuple]:
    pair_specs_by_key = {spec[0]: spec for spec in PAIR_SPECS}
    if pairs is None:
        return list(PAIR_SPECS)
    requested_pairs = list(dict.fromkeys(pairs))
    if not requested_pairs:
        raise ValueError("pairs must contain at least one pair when set.")
    unknown_pairs = [pair_key for pair_key in requested_pairs if pair_key not in pair_specs_by_key]
    if unknown_pairs:
        raise ValueError(f"Unknown prompt pairs: {unknown_pairs}")
    return [pair_specs_by_key[pair_key] for pair_key in requested_pairs]


def aggregate_similarity_table(
    result: dict,
    layer_range: tuple[int, int] | None,
    pairs: list[str] | None,
    show_std: bool,
) -> pd.DataFrame:
    aggregate_pairs = result["aggregate"]["pairs"]
    layers = resolve_plot_layers(result, layer_range=layer_range)
    pair_specs = resolve_plot_pairs(pairs)
    rows = {}
    for pair_key, _, _, _ in pair_specs:
        mean_values = np.asarray(aggregate_pairs[pair_key]["mean"], dtype=float)[layers]
        std_values = np.asarray(aggregate_pairs[pair_key]["std"], dtype=float)[layers]
        rows[pair_key] = [
            f"{mean:.4f} +/- {std:.4f}" if show_std else f"{mean:.4f}"
            for mean, std in zip(mean_values, std_values)
        ]
    return pd.DataFrame.from_dict(
        rows,
        orient="index",
        columns=[f"layer {layer}" for layer in layers],
    ).rename_axis("Pair")


def plot_similarity_aggregate(
    result: dict,
    model_key: str,
    layer_range: tuple[int, int] | None,
    pairs: list[str] | None,
    show_std: bool,
):
    aggregate_pairs = result["aggregate"]["pairs"]
    layers = resolve_plot_layers(result, layer_range=layer_range)
    pair_specs = resolve_plot_pairs(pairs)
    plot_series = []
    lower_values = []
    upper_values = []

    for pair_key, _, _, label in pair_specs:
        mean_values = np.asarray(aggregate_pairs[pair_key]["mean"], dtype=float)[layers]
        std_values = np.asarray(aggregate_pairs[pair_key]["std"], dtype=float)[layers]
        lower = np.clip(mean_values - std_values, -1.0, 1.0)
        upper = np.clip(mean_values + std_values, -1.0, 1.0)
        color = PAIR_COLORS[pair_key]
        plot_series.append((pair_key, label, color, mean_values, lower, upper))
        lower_values.extend(lower.tolist())
        upper_values.extend(upper.tolist())

    y_min = np.floor(min(lower_values) * 10) / 10
    y_max = np.ceil(max(upper_values) * 10) / 10
    if np.isclose(y_min, y_max):
        y_min = max(-1.0, y_min - 0.1)
        y_max = min(1.0, y_max + 0.1)

    figure, axis = plt.subplots(figsize=(22, 8))
    for pair_key, label, color, mean_values, lower, upper in plot_series:
        if show_std:
            axis.fill_between(layers, lower, upper, color=color, alpha=0.08, linewidth=0)
        axis.plot(layers, mean_values, color=color, linewidth=2.1, label=f"{pair_key}: {label}")

    axis.set_ylim(y_min, y_max)
    axis.set_yticks(np.round(np.arange(y_min, y_max + 0.05, 0.1), 1))
    if len(layers) == 1:
        axis.set_xlim(layers[0] - 0.5, layers[0] + 0.5)
    else:
        axis.set_xlim(layers[0], layers[-1])
    axis.set_xticks(layers)
    axis.set_xlabel("transformer layer")
    axis.set_ylabel("cosine similarity")
    axis.set_title(
        f"Representation similarity | {model_key} | "
        f"n={result['aggregate']['record_count']}",
        fontweight="bold",
    )
    axis.grid(alpha=0.22)
    axis.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="prompt pair")
    figure.tight_layout(rect=(0, 0, 0.78, 1))
    return figure, axis


def visualize_similarity(
    record_limit: int | None,
    layer_range: tuple[int, int] | None,
    pairs: list[str] | None,
    show_std: bool,
) -> None:
    output_path = similarity_output_path(record_limit)
    if not output_path.exists():
        raise FileNotFoundError(
            f"Similarity result does not exist: {output_path}. Calculate it first."
        )
    with open(output_path, encoding="utf-8") as file:
        result = json.load(file)
    display(
        aggregate_similarity_table(
            result,
            layer_range=layer_range,
            pairs=pairs,
            show_std=show_std,
        ).style
    )
    plot_similarity_aggregate(
        result,
        model_key=MODEL_KEY,
        layer_range=layer_range,
        pairs=pairs,
        show_std=show_std,
    )

## Behavior Configuration

These options are consumed only by the final manual-action cell.

In [ ]:
MODEL_KEY = "qwen2.5-7b"
# MODEL_KEY = "llama3.1-8b"
# MODEL_KEY = "qwen3.5-9b"

RECORD_LIMIT = None  # Set a positive integer for a diagnostic result file.
FILTER_BATCH_SIZE = 64  # Target checks or diagnostic prompts processed per batch.
FAILED_GENERATION_MAX_NEW_TOKENS = 32
SIMILARITY_RECORD_BATCH_SIZE = 8  # 3 prompts are analyzed per record.
LOG_FAILED_RECORDS = False
OVERWRITE_EXISTING_RESULT = False

PLOT_LAYER_RANGE = None  # None or an inclusive (start_layer, end_layer) tuple.
PLOT_PAIRS = None  # None or a list such as ["p1-p2", "p1-p3"].
PLOT_SHOW_STD = True

## Manual Actions

Uncomment one action at a time. Visualization reads only the existing JSON and never loads the model or dataset.

In [ ]:
# Calculate and save similarity JSON:
# calculate_similarity(
#     record_limit=RECORD_LIMIT,
#     filter_batch_size=FILTER_BATCH_SIZE,
#     failed_generation_max_new_tokens=FAILED_GENERATION_MAX_NEW_TOKENS,
#     similarity_record_batch_size=SIMILARITY_RECORD_BATCH_SIZE,
#     log_failed_records=LOG_FAILED_RECORDS,
#     overwrite_output=OVERWRITE_EXISTING_RESULT,
# )

# Visualize an existing similarity JSON:
# visualize_similarity(
#     record_limit=RECORD_LIMIT,
#     layer_range=PLOT_LAYER_RANGE,
#     pairs=PLOT_PAIRS,
#     show_std=PLOT_SHOW_STD,
# )